# Daily ledger and data quality

In [1]:
from pathlib import Path
import sys
import duckdb, pandas as pd, matplotlib.pyplot as plt
root=next(path for path in [Path.cwd(), *Path.cwd().parents] if (path/'config/default.json').exists())
sys.path.insert(0,str(root))
output=root/'artifacts'
figures=root/'reports/figures'
db=duckdb.connect(str(output/'operations.duckdb'),read_only=True,config={'threads':2})
def read(name):
    return pd.read_csv(output/f'{name}.csv')

In [2]:
display(db.sql("SELECT count(*) pair_days, count(DISTINCT date) AS calendar_days, count(DISTINCT sku) skus, count(DISTINCT site) sites FROM daily").df())
display(db.sql("SELECT count_if(closing<0) negative_rows, count_if(opening+received+inbound-fulfilled-outbound<>closing) out_of_balance, count_if(demand<>fulfilled+lost) demand_mismatch FROM daily").df())
display(db.sql("SELECT site,avg(utilization) mean_utilization,max(utilization) maximum_utilization FROM (SELECT date,site,sum(closing)/max(capacity) utilization FROM enriched GROUP BY 1,2) GROUP BY site ORDER BY site").df())

,pair_days,calendar_days,skus,sites
0,411000,548,150,5


,negative_rows,out_of_balance,demand_mismatch
0,0.0,0.0,0.0


,site,mean_utilization,maximum_utilization
0,0,0.251917,0.331826
1,1,0.140210,0.258624
2,2,0.135977,0.229725
3,3,0.184794,0.274429
4,4,0.183528,0.296051


The ledger checks identify unit mismatches directly. Peak receiving capacity and transaction-to-event reconciliation also run in the warehouse builder.